# TOI-4137b Analysis

This notebook computes:
1. Realignment timescales ($\tau_{\rm CE}$ and $\tau_{\rm RA}$).
2. Circularization timescale ($\tau_{\rm circ}$).

Known TOI-4137b parameters are pre-filled from your provided values, with orbital period from `run/TOI-4137b_joint_lc_rv.py`.

In [1]:
import numpy as np

YEAR_S = 365.25 * 24.0 * 3600.0
DAY_S = 24.0 * 3600.0
MJUP_PER_MSUN = 1047.348644
AU_PER_RSUN = 215.03215567054764
RJUP_PER_RSUN = 0.10045
AU_PER_RJUP = AU_PER_RSUN / RJUP_PER_RSUN
MSUN_KG = 1.98847e30
MJUP_KG = 1.89813e27
AU_M = 1.495978707e11
RSUN_M = 6.957e8
RJUP_M = RJUP_PER_RSUN * RSUN_M

rng = np.random.default_rng(4137)

def sample_asymmetric(value, plus, minus, size, rng):
    z = rng.standard_normal(size)
    sigma = np.where(z >= 0.0, plus, minus)
    x = value + z * sigma
    return np.where(x > 0.0, x, 1e-30)

def summarize_samples(x):
    q16, q50, q84 = np.percentile(x, [16, 50, 84])
    return q50, q84 - q50, q50 - q16

def mass_ratio_q(mp_mjup, mstar_msun):
    return (mp_mjup / MJUP_PER_MSUN) / mstar_msun

def a_over_rstar(a_au, rstar_rsun):
    return (a_au * AU_PER_RSUN) / rstar_rsun

def k2_from_eta_surface(eta2_surface):
    return (3.0 - eta2_surface) / (4.0 + 2.0 * eta2_surface)

def Q_from_dadt(k2, M_star_kg, R_p_m, n_rad_s, M_p_kg, a_m, da_dt_m_s):
    return (3.0 * k2 * M_star_kg * (R_p_m ** 5) * n_rad_s) / (M_p_kg * (a_m ** 5) * abs(da_dt_m_s))


## 1) Realignment Timescales

Using:

\[
\frac{1}{\tau_{\rm CE}} = \frac{1}{10 \cdot 10^{9} \;\mathrm{yr}} q^2 \left(\frac{a/R_\star}{40}\right)^{-6}
\]
\[
\frac{1}{\tau_{\rm RA}} = \frac{1}{0.25 \cdot 5 \cdot 10^{9} \;\mathrm{yr}} q^2(1+q)^{5/6}\left(\frac{a/R_\star}{6}\right)^{-17/2}
\]
where $q=M_p/M_\star$.

In [2]:
posterior = {
    "Mp_Mjup": {"value": 1.44, "plus": 0.17, "minus": 0.15},
    "Mstar_Msun": {"value": 1.313, "plus": 0.068, "minus": 0.071},
    "Rp_Rjup": {"value": 1.211, "plus": 0.040, "minus": 0.039},
    "Rstar_Rsun": {"value": 1.438, "plus": 0.039, "minus": 0.038},
    "a_AU": {"value": 0.05222, "plus": 0.00089, "minus": 0.00096},
}

period_days = 3.8016122
N = 200_000

q_c = mass_ratio_q(posterior["Mp_Mjup"]["value"], posterior["Mstar_Msun"]["value"])
aRs_c = a_over_rstar(posterior["a_AU"]["value"], posterior["Rstar_Rsun"]["value"])

inv_tau_ce_c = (1.0 / (10.0 * 1.0e9)) * (q_c ** 2) * ((aRs_c / 40.0) ** (-6.0))
inv_tau_ra_c = (1.0 / (0.25 * 5.0 * 1.0e9)) * (q_c ** 2) * ((1.0 + q_c) ** (5.0 / 6.0)) * ((aRs_c / 6.0) ** (-17.0 / 2.0))
tau_ce_c_yr = 1.0 / inv_tau_ce_c
tau_ra_c_yr = 1.0 / inv_tau_ra_c

mp_s = sample_asymmetric(**posterior["Mp_Mjup"], size=N, rng=rng)
ms_s = sample_asymmetric(**posterior["Mstar_Msun"], size=N, rng=rng)
rp_s = sample_asymmetric(**posterior["Rp_Rjup"], size=N, rng=rng)
rs_s = sample_asymmetric(**posterior["Rstar_Rsun"], size=N, rng=rng)
a_s = sample_asymmetric(**posterior["a_AU"], size=N, rng=rng)

q_s = mass_ratio_q(mp_s, ms_s)
aRs_s = a_over_rstar(a_s, rs_s)

tau_ce_s_yr = 1.0 / ((1.0 / (10.0 * 1.0e9)) * (q_s ** 2) * ((aRs_s / 40.0) ** (-6.0)))
tau_ra_s_yr = 1.0 / ((1.0 / (0.25 * 5.0 * 1.0e9)) * (q_s ** 2) * ((1.0 + q_s) ** (5.0 / 6.0)) * ((aRs_s / 6.0) ** (-17.0 / 2.0)))

ce50, ce_p, ce_m = summarize_samples(tau_ce_s_yr)
ra50, ra_p, ra_m = summarize_samples(tau_ra_s_yr)

print(f"q = {q_c:.6e}")
print(f"a/R* = {aRs_c:.4f}")
print(f"tau_CE (central): {tau_ce_c_yr:.3e} yr ({tau_ce_c_yr/1e9:.3e} Gyr)")
print(f"tau_RA (central): {tau_ra_c_yr:.3e} yr ({tau_ra_c_yr/1e9:.3e} Gyr)")
print(f"tau_CE (16/50/84): {ce50:.3e} +{ce_p:.3e} / -{ce_m:.3e} yr")
print(f"tau_RA (16/50/84): {ra50:.3e} +{ra_p:.3e} / -{ra_m:.3e} yr")

q = 1.047144e-03
a/R* = 7.8087
tau_CE (central): 5.048e+11 yr (5.048e+02 Gyr)
tau_RA (central): 1.069e+16 yr (1.069e+07 Gyr)
tau_CE (16/50/84): 5.000e+11 +1.835e+11 / -1.332e+11 yr
tau_RA (16/50/84): 1.059e+16 +4.702e+15 / -3.231e+15 yr


## 2) Circularization Timescale

Using:
\[
\tau_{\rm circ} = \frac{2}{81}\frac{Q_p'}{n}\frac{M_p}{M_*}\left(\frac{a}{R_p}\right)^5\left[\frac{Q_p'}{Q_*'}\left(\frac{M_p}{M_*}\right)^2\left(\frac{R_*}{R_p}\right)^5 F_* + F_p\right]^{-1}
\]
with $Q' = 3Q/(2k_2)$ and $n=2\pi/P$.

For the Love number relation:
\[
k_2 = \frac{3 - \eta_2(R)}{4 + 2\eta_2(R)}
\]
you can either supply $\eta_2(R)$ or set $k_2$ directly.

In [ ]:
k2_planet = 0.379
k2_star = 0.01444298
Q_planet = 1.0e6
Q_star = 1.0e7
F_planet = 1.0
F_star = 1.0
eta2_surface_planet = None
eta2_surface_star = None

if eta2_surface_planet is not None:
    k2_planet = k2_from_eta_surface(eta2_surface_planet)
if eta2_surface_star is not None:
    k2_star = k2_from_eta_surface(eta2_surface_star)

if min(k2_planet, k2_star, Q_planet, Q_star) <= 0.0:
    raise ValueError("k2 and Q inputs must be positive.")

Qp_prime = 3.0 * Q_planet / (2.0 * k2_planet)
Qs_prime = 3.0 * Q_star / (2.0 * k2_star)

def tau_circ_seconds(mp_mjup, mstar_msun, rp_rjup, rstar_rsun, a_au, p_days, Qp_p, Qs_p, Fp=1.0, Fs=1.0):
    q = mass_ratio_q(mp_mjup, mstar_msun)
    n = 2.0 * np.pi / (p_days * DAY_S)
    a_over_rp = (a_au * AU_PER_RJUP) / rp_rjup
    rstar_over_rp = (rstar_rsun / RJUP_PER_RSUN) / rp_rjup
    bracket = (Qp_p / Qs_p) * (q ** 2) * (rstar_over_rp ** 5) * Fs + Fp
    return (2.0 / 81.0) * (Qp_p / n) * q * (a_over_rp ** 5) / bracket

tau_circ_c_s = tau_circ_seconds(
    posterior["Mp_Mjup"]["value"],
    posterior["Mstar_Msun"]["value"],
    posterior["Rp_Rjup"]["value"],
    posterior["Rstar_Rsun"]["value"],
    posterior["a_AU"]["value"],
    period_days,
    Qp_prime,
    Qs_prime,
    F_planet,
    F_star,
)

tau_circ_s_samp = tau_circ_seconds(mp_s, ms_s, rp_s, rs_s, a_s, period_days, Qp_prime, Qs_prime, F_planet, F_star)
tc50_s, tc_p_s, tc_m_s = summarize_samples(tau_circ_s_samp)

print(f"k2_planet={k2_planet:.5g}, Q_planet={Q_planet:.3e}, Qp'={Qp_prime:.3e}")
print(f"k2_star={k2_star:.5g}, Q_star={Q_star:.3e}, Q*'={Qs_prime:.3e}")
print(f"tau_circ (central): {tau_circ_c_s / YEAR_S:.3e} yr ({tau_circ_c_s / YEAR_S / 1e9:.3e} Gyr)")
print(f"tau_circ (16/50/84): {tc50_s / YEAR_S:.3e} +{tc_p_s / YEAR_S:.3e} / -{tc_m_s / YEAR_S:.3e} yr")

k2_planet=0.379, Q_planet=1.000e+06, Qp'=3.958e+06
k2_star=0.014443, Q_star=1.000e+05, Q*'=1.039e+07
tau_circ (central): 1.036e+09 yr (1.036e+00 Gyr)
tau_circ (16/50/84): 1.033e+09 +2.247e+08 / -1.890e+08 yr


In [4]:
da_dt_m_per_s = None

if da_dt_m_per_s is not None:
    n_c = 2.0 * np.pi / (period_days * DAY_S)
    q_planet_from_dadt = Q_from_dadt(
        k2_planet,
        posterior["Mstar_Msun"]["value"] * MSUN_KG,
        posterior["Rp_Rjup"]["value"] * RJUP_M,
        n_c,
        posterior["Mp_Mjup"]["value"] * MJUP_KG,
        posterior["a_AU"]["value"] * AU_M,
        da_dt_m_per_s,
    )
    print(f"Q from da/dt: {q_planet_from_dadt:.3e}")
else:
    print("Set da_dt_m_per_s to a value (m/s) to compute Q from your da/dt equation.")

Set da_dt_m_per_s to a value (m/s) to compute Q from your da/dt equation.


If you want to use a different tidal model, edit the input cell values (`k2_planet`, `Q_planet`, `k2_star`, `Q_star`, `F_planet`, `F_star`, or `period_days`) and re-run the calculation cells.